In [ ]:
!pip uninstall -y paddleocr paddlepaddle numpy
!pip install "numpy<2.0.0" paddlepaddle==3.3.1 paddleocr==2.7.3
!pip install opencv-python-headless matplotlib transformers shapely pyclipper torch torchvision torchaudio langchain-text-splitters

import os
os.environ['FLAGS_use_onednn'] = '0'
import numpy as np
print(f"Verified NumPy version: {np.__version__}")

In [ ]:
import os

# CREATE PROJECT DIRECTORY STRUCTURE
folders = [
    "data/raw_images",
    "outputs/visualizations",
    "outputs/OCR_results"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

In [ ]:
from google.colab import files
import os

# PRODUCT IMAGES UPLOAD
print("Please select the product label images you want to process:")
uploaded = files.upload()

for filename in uploaded.keys():
    target_path = f"data/raw_images/{filename}"
    with open(target_path, 'wb') as f:
        f.write(uploaded[filename])

print(f"Successfully uploaded {len(uploaded)} images to data/raw_images/")

In [ ]:
import cv2
import os

def apply_5_module_preprocessing(image_path, output_path):
    # 1. Load and Denoise (Non-Local Means)
    img = cv2.imread(image_path)
    if img is None: return
    denoised = cv2.fastNlMeansDenoisingColored(img, None, h=10, hColor=10, templateWindowSize=7, searchWindowSize=21)

    # 2. CLAHE (Contrast Enhancement)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)

    # 3. Unsharp Masking (Sharpening)
    blurred = cv2.GaussianBlur(enhanced, (0, 0), 2.0)
    sharpened = cv2.addWeighted(enhanced, 1.8, blurred, -0.8, 0)


    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, sharpened)
    return output_path

raw_dir = "data/raw_images"
proc_dir = "data/dl_preprocessed"

for img_file in os.listdir(raw_dir):
    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
        apply_5_module_preprocessing(os.path.join(raw_dir, img_file), os.path.join(proc_dir, img_file))

print(f"Preprocessing complete. Optimized images ready in {proc_dir}")

In [ ]:
import subprocess
import sys

# Install Real-ESRGAN and compatible basicsr version
subprocess.run(['pip', 'install', 'realesrgan', 'gdown', 'basicsr'])

import torchvision
import torchvision.transforms.functional as F
sys.modules['torchvision.transforms.functional_tensor'] = F

import cv2
import torch
import numpy as np
import os
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from PIL import Image

print("Preprocessing dependencies loaded and patched.")


model_path = "RealESRGAN_x4plus.pth"

if not os.path.exists(model_path):
    print("Downloading Real-ESRGAN weights...")
    subprocess.run([
        'gdown',
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        '-O', model_path
    ])
    print("Weights downloaded.")
else:
    print("Weights already exist. Skipping download.")


sr_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23,
    num_grow_ch=32, scale=4
)

upsampler = RealESRGANer(
    scale=4,
    model_path=model_path,
    model=sr_model,
    tile=0,
    half=True if torch.cuda.is_available() else False
)

print(f"Real-ESRGAN loaded on {'GPU' if torch.cuda.is_available() else 'CPU'}.")


proc_dir = "data/dl_preprocessed"

processed_files = [
    f for f in os.listdir(proc_dir)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]

if not processed_files:
    print("No classically preprocessed images found. Run the preprocessing cell first.")
else:
    print(f"\nApplying Real-ESRGAN to {len(processed_files)} image(s)...\n")

    for filename in processed_files:
        img_path = os.path.join(proc_dir, filename)


        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"  Skipping {filename}: unreadable.")
            continue

        try:

            original_h, original_w = img.shape[:2]

            enhanced, _ = upsampler.enhance(img, outscale=4)


            enhanced_resized = cv2.resize(
                enhanced,
                (original_w, original_h),
                interpolation=cv2.INTER_LANCZOS4
            )


            cv2.imwrite(img_path, enhanced_resized)
            print(f"  Done: {filename}")

        except Exception as e:
            print(f"  Error on {filename}: {e}. Skipping.")

    print("\nReal-ESRGAN enhancement complete.")
    print("Images in data/dl_preprocessed now have both classical + DL preprocessing applied.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os


raw_dir = 'data/raw_images'
proc_dir = 'data/dl_preprocessed'


files = [f for f in os.listdir(raw_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if files:

    latest_file = max([os.path.join(raw_dir, f) for f in files], key=os.path.getmtime)
    filename = os.path.basename(latest_file)

    raw_path = latest_file
    proc_path = os.path.join(proc_dir, filename)

    if os.path.exists(raw_path) and os.path.exists(proc_path):
        raw_img = cv2.imread(raw_path)
        proc_img = cv2.imread(proc_path)

        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        proc_img = cv2.cvtColor(proc_img, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(20, 10))

        axes[0].imshow(raw_img)
        axes[0].set_title(f"Original: {filename}")
        axes[0].axis('off')

        axes[1].imshow(proc_img)
        axes[1].set_title(f"Preprocessed: {filename}")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print(f"Error: Preprocessed version of {filename} not found. Ensure preprocessing cell has run.")
else:
    print("No images found in data/raw_images. Please upload an image using the upload cell.")

In [ ]:
import paddle
import sys
import os


if not hasattr(paddle.base.libpaddle.AnalysisConfig, 'set_optimization_level'):
    def set_optimization_level(self, level):
        pass
    paddle.base.libpaddle.AnalysisConfig.set_optimization_level = set_optimization_level


def get_ocr_engine():
    global ocr
    if 'ocr' in globals():
        print(" PaddleOCR is already active in memory.")
        return ocr

    try:

        import paddlex
        from paddlex.repo_manager import _GlobalContext
        _GlobalContext._initialized = True
    except Exception:
        pass

    try:
        from paddleocr import PaddleOCR
        ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True, show_log=False)
        print("PaddleOCR engine initialized successfully.")
        return ocr
    except Exception as e:
        if "PDX has already been initialized" in str(e):

            try:
                from paddleocr import PaddleOCR
                ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True, show_log=False)
                print(" Engine recovered from lock.")
                return ocr
            except:
                print("Engine is locked but active. Proceeding.")
        else:
            print(f" Unexpected Error: {e}")

# Run the secure initialization
ocr = get_ocr_engine()

In [ ]:

import os, cv2, numpy as np
from PIL import Image as PILImage


dl_folder = "data/dl_preprocessed"
raw_folder = "data/raw_images"
os.makedirs(dl_folder, exist_ok=True)


if 'ocr' not in globals() or ocr is None:
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True, show_log=False)


target_ocr_paths = [os.path.join(dl_folder, f) for f in os.listdir(dl_folder)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
                    and os.path.isfile(os.path.join(dl_folder, f))]
target_ocr_paths.sort()

all_results, all_images_detected_text = [], []
final_valid_paths = []

if not target_ocr_paths:
    print("No preprocessed images found. Please run the preprocessing cell first.")
else:
    for path in target_ocr_paths:
        img_check = cv2.imread(path)
        if img_check is None:
            print(f"Skipping {os.path.basename(path)}: File is empty or unreadable.")
            continue

        print(f"\nProcessing: {os.path.basename(path)}")
        try:
            result = ocr.ocr(path, cls=True)


            final_valid_paths.append(path)

            if result and result[0]:
                all_results.append(result)
                txts = [line[1][0] for line in result[0]]
                all_images_detected_text.extend(txts)
                print(f"Extracted {len(txts)} text blocks.")
            else:
                all_results.append(None)
                print("No text detected in this image.")

        except Exception as e:
            print(f"Error processing {os.path.basename(path)}: {e}")

            final_valid_paths.pop()


target_ocr_paths = final_valid_paths
print(f"\nTotal images processed: {len(target_ocr_paths)}")

In [ ]:

import os

if 'all_results' in locals() and 'target_ocr_paths' in locals():
    print(f"Successfully processed {len(all_results)} images.")


    for i, (path, result) in enumerate(zip(target_ocr_paths, all_results)):
        count = len(result[0]) if result and result[0] else 0
        print(f"Image {i+1}: {os.path.basename(path)} - {count} text blocks found.")
else:
    print("No results found. Please ensure the OCR extraction cell has run successfully.")

In [ ]:

if 'all_images_detected_text' in locals() and all_images_detected_text:
    print(f"\n Total Text Blocks Extracted: {len(all_images_detected_text)} ")
    for i, text_item in enumerate(all_images_detected_text):
        print(f"{i+1}: {text_item}")
else:
    print("No text extracted. Please run the processing cell (r7L-CazvAmCy) first.")

In [ ]:

if 'all_results' in locals() and all_results:
    print(" Extracted Text per Image \n")

    for i, (path, result) in enumerate(zip(target_ocr_paths, all_results)):
        image_name = os.path.basename(path)
        print(f"Image {i+1} ({image_name}):")

        if result and result[0]:

            image_text = " ".join([line[1][0] for line in result[0]])
            print(f'"{image_text}"')
        else:
            print('"No text detected."')
        print("-" * 30)
else:
    print("No OCR results available. Please run the processing cell (r7L-CazvAmCy) first.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

raw_folder = "data/raw_images"
dl_folder = "data/dl_preprocessed"


viz_paths = [f for f in os.listdir(dl_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
viz_paths.sort()

if viz_paths:
    print(f"--- Comparing {len(viz_paths)} Uploads with DL Preprocessing Results ---")

    for i, filename in enumerate(viz_paths):
        raw_path = os.path.join(raw_folder, filename)
        proc_path = os.path.join(dl_folder, filename)

        raw_img = cv2.imread(raw_path)
        proc_img = cv2.imread(proc_path)

        if raw_img is None or proc_img is None:
            continue

        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        proc_img = cv2.cvtColor(proc_img, cv2.COLOR_BGR2RGB)


        if 'all_results' in locals() and i < len(all_results):
            result = all_results[i]
            if result and result[0]:
                for line in result[0]:
                    points = np.array(line[0]).astype(np.int32)
                    cv2.polylines(proc_img, [points], isClosed=True, color=(0, 255, 0), thickness=3)

        # side-by-side plot
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(raw_img)
        axes[0].set_title(f"ORIGINAL UPLOAD: {filename}")
        axes[0].axis('off')

        axes[1].imshow(proc_img)
        has_text = "Detections Found" if ('all_results' in locals() and i < len(all_results) and all_results[i] and all_results[i][0]) else "No Text Detected"
        axes[1].set_title(f"PREPROCESSED + OCR: {filename} ({has_text})")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()
else:
    print("No images found to visualize.")

In [ ]:
!pip install -q -U transformers accelerate einops timm sentencepiece

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import notebook_login


MODEL_PATH = "OpenGVLab/InternVL3_5-1B-HF"

print("Loading InternVL3.5-1B (HF native version)...")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval()

print(f"Model loaded successfully on {device}.")

In [ ]:
import time
import os
from PIL import Image

def post_process_with_internvl(raw_text, image_path=None, max_retries=3):

    prompt_text = f"""
You are an expert OCR correction and food label analysis system with an experience of 15 years.

You are given:

1. A PREPROCESSED product label image.
2. Raw OCR text extracted from the same image.

Instructions:

 Use the image as the primary source of truth.
 Use OCR text as supporting evidence.
 Correct OCR mistakes.
 Preserve all numbers exactly.
 Preserve nutrition values exactly.
 Preserve ingredient names exactly.
 Do not hallucinate information.
 If text is unreadable, mark it as [uncertain].
 Now Generate a structured Markdown report with:
 Product Summary
 Directions
 Ingredients
 Allergy Advice & Cautions
 Contact / Manufacturer Information

RAW OCR TEXT:
{raw_text}
"""

    image_obj = None

    if image_path and os.path.exists(image_path):
        try:
            image_obj = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"  [Warning] Could not load image: {e}")

    content = []

    if image_obj is not None:
        content.append({"type": "image"})

    content.append({
        "type": "text",
        "text": prompt_text
    })

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    for attempt in range(max_retries):
        try:
            return call_internvl(
                messages,
                image_obj=image_obj,
                max_new_tokens=1024
            )

        except Exception as e:
            wait = 5 * (2 ** attempt)
            print(f"Error: {e}")
            print(f"Retrying in {wait}s...")
            time.sleep(wait)

    return "Error: Failed after multiple retries."

In [ ]:
import time
import os
from PIL import Image

def extract_ingredients_with_internvl(raw_text, image_path=None, max_retries=3):

    prompt = f"""
 Assume yourself to be a NLP engineer and OCR SYSTEM having an experience of 15 years and You are given:

1. A PREPROCESSED product-label image.
2. Raw OCR text extracted from the same image.

Instructions:
 Use the image as the primary source of truth.
 Cross-check against OCR text.
 Correct OCR mistakes.
 Extract ONLY the ingredients list.
 Do not include nutrition values.
 Do not include directions.
 Do not include manufacturer details.

Return ONLY a clean Markdown bullet list.

RAW OCR TEXT:
{raw_text}
"""

    image_obj = None

    if image_path and os.path.exists(image_path):
        try:
            image_obj = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"  [Warning] Could not load image: {e}")

    content = []

    if image_obj is not None:
        content.append({"type": "image"})

    content.append({
        "type": "text",
        "text": prompt
    })

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    for attempt in range(max_retries):
        try:
            return call_internvl(
                messages,
                image_obj=image_obj,
                max_new_tokens=512
            )

        except Exception as e:
            wait = 10 * (2 ** attempt)
            print(f"Error: {e}")
            print(f"Retry {attempt+1}/{max_retries} in {wait}s...")
            time.sleep(wait)

    return "Error: Failed after multiple retries."

In [ ]:
import torch

def call_internvl(messages, image_obj=None, max_new_tokens=1024):
    """
    Sends image + text together to the model.
    """

    text_prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    if image_obj is not None:
        inputs = processor(
            text=text_prompt,
            images=image_obj,
            return_tensors="pt"
        ).to(model.device, dtype=dtype)

        print("[Multimodal] Image + text both sent to InternVL")

    else:
        inputs = processor(
            text=text_prompt,
            return_tensors="pt"
        ).to(model.device, dtype=dtype)

        print("[Text-only] No image provided")

    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )

    output = processor.decode(
        generate_ids[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return output

In [ ]:
import os
from IPython.display import Markdown, display

structured_results_per_image = {}

if 'target_ocr_paths' in locals() and 'all_results' in locals() and target_ocr_paths:

    print(
        f"Processing {len(target_ocr_paths)} images individually with InternVL (PREPROCESSED IMAGE + OCR TEXT)...\n"
    )

    for path, result in zip(target_ocr_paths, all_results):

        image_name = os.path.basename(path)

        preprocessed_image_path = path

        if result and result[0]:
            raw_text = " ".join(
                [line[1][0] for line in result[0]]
            )
        else:
            raw_text = ""

        print("*" * 60)
        print(f"Image: {image_name}")
        print("*" * 60)

        if not raw_text.strip():
            print("No text detected in this image.\n")
            continue

        report = post_process_with_internvl(
            raw_text,
            image_path=preprocessed_image_path
        )

        structured_results_per_image[image_name] = report

        display(
            Markdown(
                f" Structured Report — {image_name}"
            )
        )

        display(Markdown(report))
        print()

    print("All images processed successfully.")

else:
    print(
        "No OCR results available. Please run OCR first."
    )

In [ ]:
import os
from IPython.display import Markdown, display

ingredients_results_per_image = {}

if 'target_ocr_paths' in locals() and 'all_results' in locals() and target_ocr_paths:

    print(
        f"Extracting ingredients from {len(target_ocr_paths)} images...\n"
    )
    for path, result in zip(target_ocr_paths, all_results):
        image_name = os.path.basename(path)
        preprocessed_image_path = path
        if result and result[0]:
            raw_text = " ".join(
                [line[1][0] for line in result[0]]
            )
        else:
            raw_text = ""
        print("*" * 60)
        print(f"Image: {image_name}")
        print("*" * 60)

        if not raw_text.strip():
            print("No text detected.\n")
            ingredients_results_per_image[image_name] = "No text detected."
            continue

        ingredients = extract_ingredients_with_internvl(
            raw_text,
            image_path=preprocessed_image_path
        )
        ingredients_results_per_image[image_name] = ingredients
        display(
            Markdown(
                f" Ingredients — {image_name}"
            )
        )
        display(Markdown(ingredients))
        print()
    print(
        "All images processed successfully."
    )
else:
    print(
        "No OCR results found."
    )

In [ ]:
import csv
import os
from google.colab import files


if 'ingredients_results_per_image' in locals() and ingredients_results_per_image:

    csv_path = "outputs/OCR_results/ingredients_output.csv"
    os.makedirs("outputs/OCR_results", exist_ok=True)

    with open(csv_path, mode='w', newline='', encoding='utf-8') as csvfile:

        writer = csv.writer(csvfile)


        writer.writerow(["Image Name", "Extracted Ingredients"])

        for image_name, ingredients_text in ingredients_results_per_image.items():


            clean_text = ingredients_text.replace("- ", "").replace("* ", "").replace("• ", "")

            writer.writerow([image_name, clean_text])

    print(f"CSV saved: {csv_path}")
    print(f"Total images saved: {len(ingredients_results_per_image)}\n")

    # Preview what was saved
    for image_name, ingredients_text in ingredients_results_per_image.items():
        print(f"  {image_name}:")
        print(f"  {ingredients_text[:100]}...")
        print()


    files.download(csv_path)

else:
    print("No ingredients data found.")
    print("Please run the ingredients extraction cell first.")

In [ ]:
import shutil
import os

# 1. CLEANUP DATA FOLDER
data_folders = ['data/raw_images', 'data/dl_preprocessed']

for folder in data_folders:
    if os.path.exists(folder):
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

print("Data folders (raw_images, dl_preprocessed) have been cleared.")

In [ ]:
import shutil
import os

# 2. CLEANUP OUTPUTS FOLDER
output_folders = ['outputs/visualizations', 'outputs/OCR_results']

for folder in output_folders:
    if os.path.exists(folder):
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

print("Output folders (visualizations, OCR_results) have been cleared.")